# 04 — Exportar o modelo para ONNX

Carrega o melhor checkpoint treinado, exporta para **ONNX** (opset 17, batch dinamico) e **valida a paridade** PyTorch vs ONNX Runtime no test set.

**Pre-requisito:** ter treinado o modelo (existe `artifacts/checkpoints/best_<modelo>.pth`).

## 0. Ambiente (Colab)

In [ ]:
import sys
from pathlib import Path

# Em Colab: instala dependencias, clona o repo e (opcional) monta o Drive
if 'google.colab' in sys.modules:
    !pip install -q timm albumentations scikit-learn pandas matplotlib seaborn tensorboard tqdm pyyaml onnx onnxruntime
    !git clone https://github.com/SEU_USUARIO/tcc-ze-praga-model-playground.git
    %cd tcc-ze-praga-model-playground
    from google.colab import drive
    drive.mount('/content/drive')

print('Setup OK')

## 1. Imports e configuracao

In [ ]:
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path('.').resolve()))
from src.utils.config import load_model_config
from src.utils.seed import set_seed
from src.data.transforms import get_val_transforms
from src.data.dataset import create_dataloaders
from src.models.factory import build_model
from src.export.to_onnx import export_to_onnx
from src.export.validate_onnx import validate_onnx

cfg = load_model_config(Path('configs/resnet50.yaml'), Path('configs/base.yaml'))
set_seed(cfg['seed'])
model_name  = cfg['model']['name']
input_size  = cfg['model']['input_size']
num_classes = cfg['num_classes']
DATA_DIR = Path('data/processed')
CKPT = Path('artifacts/checkpoints') / ('best_' + model_name + '.pth')
print('Modelo:', model_name, '| checkpoint:', CKPT, '| existe?', CKPT.exists())

## 2. Carregar o modelo treinado

In [ ]:
# Export sempre em CPU para portabilidade
model = build_model(model_name, num_classes=num_classes, pretrained=False)
model.load_state_dict(torch.load(CKPT, map_location='cpu'))
model.eval()
print('Pesos carregados.')

## 3. Exportar para ONNX

In [ ]:
onnx_path = Path('artifacts/onnx') / (model_name + '.onnx')
export_to_onnx(model, onnx_path, input_size=input_size)
print('ONNX salvo em:', onnx_path)

## 4. Validar paridade (PyTorch vs ONNX Runtime)

In [ ]:
val_tf = get_val_transforms(input_size)
_, _, test_loader = create_dataloaders(
    processed_dir=DATA_DIR,
    train_transform=val_tf,
    val_transform=val_tf,
    batch_size=cfg['batch_size'],
    num_workers=0,
)
passed = validate_onnx(model, onnx_path, test_loader, device=torch.device('cpu'))
print('Paridade OK?', passed)

## 5. Proximo passo

Copie o arquivo `artifacts/onnx/<modelo>.onnx` para o backend e ligue o `InferenceService` ao modelo real (TCC-023).

In [ ]:
print('Pronto. Arquivo ONNX:', (Path('artifacts/onnx') / (model_name + '.onnx')).resolve())